<a href="https://colab.research.google.com/github/aliffaimanrushlan07/SCOPE/blob/claude%2Fbusy-hamilton-58opcv/ml/notebooks/finetune_vit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCOPE — fine-tuning the Vision Transformer

Trains the evidence classifier used at **S10**: given a photograph, return
P(smoking violation).

**Run this in Google Colab with a GPU runtime.** Runtime → Change runtime type →
T4 GPU. On CPU a single epoch takes hours; on a T4 it takes minutes.

### Why KerasHub and not HuggingFace `transformers`

`transformers` **removed TensorFlow support** — `TFViTForImageClassification`
no longer exists. KerasHub is the Keras-native replacement, runs on the
TensorFlow backend, and keeps the project honest to Table 3.5 of the report
(*TensorFlow / Keras*). Nothing here needs PyTorch.

### Before you start

Run `python ml/prepare_dataset.py` locally first. This notebook consumes the
`manifest.csv` it produces — the split is decided there, deliberately, so that
it is reproducible and free of train/test leakage rather than re-rolled here.


## 1. Environment


In [ ]:
!pip install -q keras-hub keras --upgrade

import os
os.environ['KERAS_BACKEND'] = 'tensorflow'   # Table 3.5 commits to TensorFlow

import json, pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_hub

print('keras     ', keras.__version__)
print('tensorflow', tf.__version__)
print('GPU       ', tf.config.list_physical_devices('GPU') or 'NONE — switch runtime to T4 GPU')


## 2. Data

Upload a zip containing `manifest.csv` and the `data/raw/` tree, or mount Drive.
The manifest's `path` column is relative to the `ml/` directory.


In [ ]:
from google.colab import files

# Option A — upload a zip of the ml/data directory
uploaded = files.upload()          # choose e.g. scope-data.zip
!unzip -q -o $(ls *.zip | head -1) -d /content/ml

# Option B — Google Drive instead (comment out Option A above)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/scope/ml' /content/ml

ML_DIR = pathlib.Path('/content/ml')
manifest = pd.read_csv(ML_DIR / 'data' / 'manifest.csv')
print(manifest.groupby(['split', 'label']).size())


### Verify the split before training

Cheap, and it protects the headline result: if a perceptual hash appeared in
two splits, the reported test accuracy would be inflated by memorisation.


In [ ]:
spanning = manifest.groupby('phash')['split'].nunique()
assert (spanning <= 1).all(), 'LEAKAGE: a hash group spans multiple splits'
assert manifest['path'].is_unique, 'duplicate paths in manifest'
print('No leakage between splits.')
print('Total images:', len(manifest))


## 3. Input pipeline

Augmentation is applied to the **training split only** — augmenting validation
or test would measure the model against images it will never see in the field.
The evaluation splits are also left unshuffled, because `score_split` in
section 8 pairs their scores with the manifest by position.

The recipe here is deliberately much heavier than the flip-and-jitter it
replaces. Two reasons:

**A ViT-Base on 1,358 images overfits.** Transformers have no built-in notion
that neighbouring pixels are related, so they need far more augmentation than a
CNN to learn that from data. This is the finding DeiT (arXiv:2012.12877) is
built on.

**The training images are not what the app will receive.** Web photographs are
close, sharp and well lit. A report from a food court is further away, shakier,
dimmer and more compressed. `RandomResizedCrop` rehearses the distance;
blur, downscale and JPEG rehearse the camera.

Written with `tf.image` rather than `keras_cv`, so nothing is added to the
TensorFlow stack Table 3.5 commits to. Mixup and CutMix are **not** included:
they are the other half of the DeiT recipe, but they need a second look at the
loss and are worth measuring separately.

There is no vertical flip: a cigarette has an orientation, and an upside-down
photograph is not a case this app will ever encounter.

In [ ]:
IMAGE_SIZE = 224          # ViT base patch16 expects 224x224
BATCH_SIZE = 32
CLASS_NAMES = ['notsmoking', 'smoking']   # index 1 == positive class
NUM_CLASSES = len(CLASS_NAMES)
POSITIVE_INDEX = CLASS_NAMES.index('smoking')
AUTOTUNE = tf.data.AUTOTUNE

# Labels are ONE-HOT, not integer indices.
#
# This is forced by label smoothing in cell 10: Keras implements it only on
# CategoricalCrossentropy, which needs one-hot targets.
# SparseCategoricalCrossentropy has no label_smoothing argument.
#
# It also makes the precision/recall/PR-AUC metrics straightforward, since they
# read the positive-class column directly.


def decode(path, label):
    """Read an image at its stored resolution — resizing happens later.

    Deliberately NOT resized to 224 here. RandomResizedCrop below takes its
    crop from the image as stored (long side 256, from make_colab_bundle.py),
    so the crop keeps detail that a pre-resize to 224 would have thrown away.
    """
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    return tf.cast(img, tf.float32), tf.one_hot(label, NUM_CLASSES)


def resize_only(img, label):
    """The evaluation path: no augmentation, just the size the model wants."""
    return tf.image.resize(img, (IMAGE_SIZE, IMAGE_SIZE)), label


# ---------------------------------------------------------------------------
# Augmentation
#
# The previous recipe was a horizontal flip plus brightness/contrast/saturation
# jitter. That is a light CNN-era pipeline, and a ViT-Base on 1,358 images
# overfits through it — the DeiT paper (arXiv:2012.12877) exists because
# transformers need far more augmentation than CNNs on small datasets.
#
# It also failed to rehearse the domain gap ml/README.md names honestly: real
# submissions are photographed FURTHER AWAY, with MORE MOTION BLUR and WORSE
# LIGHT than the web-sourced training images. Augmentation that does not
# reproduce that gap cannot help the model close it.
#
# Written with tf.image rather than keras_cv, so the pipeline adds no
# dependency beyond the TensorFlow stack Table 3.5 commits to.
# ---------------------------------------------------------------------------

RANDAUGMENT_OPS = 2       # ops sampled per image
RANDAUGMENT_MAGNITUDE = 0.5
DEGRADE_PROBABILITY = 0.5  # chance of one phone-camera degradation
ERASE_PROBABILITY = 0.25


def random_resized_crop(img, min_scale=0.6, max_scale=1.0):
    """Crop 60-100% of the area at a random aspect, then resize to 224.

    The single most valuable item in this pipeline. A crop is a zoom, so
    training on crops teaches the model that a violation may occupy a small or
    a large part of the frame — which is exactly the difference between a
    web photograph and one taken across a food court.
    """
    shape = tf.cast(tf.shape(img)[:2], tf.float32)
    area = shape[0] * shape[1]

    target_area = area * tf.random.uniform([], min_scale, max_scale)
    log_ratio = tf.random.uniform([], tf.math.log(3 / 4), tf.math.log(4 / 3))
    aspect = tf.exp(log_ratio)

    crop_h = tf.minimum(tf.sqrt(target_area / aspect), shape[0])
    crop_w = tf.minimum(tf.sqrt(target_area * aspect), shape[1])
    crop_h = tf.maximum(tf.cast(crop_h, tf.int32), 1)
    crop_w = tf.maximum(tf.cast(crop_w, tf.int32), 1)

    offset_y = tf.random.uniform([], 0, tf.shape(img)[0] - crop_h + 1, tf.int32)
    offset_x = tf.random.uniform([], 0, tf.shape(img)[1] - crop_w + 1, tf.int32)

    crop = img[offset_y:offset_y + crop_h, offset_x:offset_x + crop_w, :]
    return tf.image.resize(crop, (IMAGE_SIZE, IMAGE_SIZE))


def _photometric_op(img, index, magnitude):
    """One RandAugment-style colour operation, chosen by index.

    A hand-rolled policy in the spirit of RandAugment (arXiv:1909.13719): N
    operations sampled uniformly at a shared magnitude, rather than three
    hand-tuned constants that have to be defended individually in the report.
    It is NOT the published op list — the geometric and PIL-specific ops
    (shear, translate, equalize, autocontrast) are left out.
    """
    sign = tf.where(tf.random.uniform([]) < 0.5, -1.0, 1.0)
    signed = magnitude * sign

    def brightness():
        return img + signed * 60.0

    def contrast():
        return tf.image.adjust_contrast(img, 1.0 + signed * 0.6)

    def saturation():
        return tf.image.adjust_saturation(img, 1.0 + signed * 0.7)

    def hue():
        return tf.image.adjust_hue(img / 255.0, signed * 0.12) * 255.0

    def posterize():
        bits = 8.0 - tf.round(magnitude * 4.0)
        levels = tf.pow(2.0, bits)
        return tf.floor(img / 256.0 * levels) / levels * 256.0

    def solarize():
        threshold = 255.0 * (1.0 - magnitude)
        return tf.where(img >= threshold, 255.0 - img, img)

    def desaturate():
        grey = tf.image.rgb_to_grayscale(img)
        return img + (tf.tile(grey, [1, 1, 3]) - img) * magnitude

    return tf.switch_case(index, [
        brightness, contrast, saturation, hue, posterize, solarize, desaturate,
    ])


def randaugment(img, n=RANDAUGMENT_OPS, magnitude=RANDAUGMENT_MAGNITUDE):
    for _ in range(n):
        index = tf.random.uniform([], 0, 7, dtype=tf.int32)
        strength = tf.random.uniform([], 0.0, magnitude)
        img = _photometric_op(img, index, strength)
        img = tf.clip_by_value(img, 0.0, 255.0)
    return img


def gaussian_blur(img):
    """Separable Gaussian blur — out-of-focus and low-light phone photographs."""
    sigma = tf.random.uniform([], 0.6, 2.0)
    offsets = tf.cast(tf.range(-3, 4), tf.float32)
    kernel = tf.exp(-tf.square(offsets) / (2.0 * tf.square(sigma)))
    kernel = kernel / tf.reduce_sum(kernel)

    horizontal = tf.tile(tf.reshape(kernel, [1, 7, 1, 1]), [1, 1, 3, 1])
    vertical = tf.tile(tf.reshape(kernel, [7, 1, 1, 1]), [1, 1, 3, 1])

    batched = img[tf.newaxis, ...]
    batched = tf.nn.depthwise_conv2d(batched, horizontal, [1, 1, 1, 1], 'SAME')
    batched = tf.nn.depthwise_conv2d(batched, vertical, [1, 1, 1, 1], 'SAME')
    return batched[0]


def motion_blur(img):
    """Directional blur — the reporter's hand moved, or the subject did.

    This is the degradation the current model has never seen, and the one a
    photograph taken quickly and discreetly is most likely to carry.
    """
    size = 9
    line = tf.ones([size], tf.float32)
    zeros = tf.zeros([size, size], tf.float32)
    middle = size // 2

    horizontal = tf.tensor_scatter_nd_update(zeros, [[middle]], [line])
    vertical = tf.transpose(horizontal)
    diagonal = tf.eye(size)
    anti_diagonal = tf.reverse(diagonal, axis=[1])

    index = tf.random.uniform([], 0, 4, dtype=tf.int32)
    kernel = tf.switch_case(index, [
        lambda: horizontal, lambda: vertical,
        lambda: diagonal, lambda: anti_diagonal,
    ])
    kernel = kernel / tf.reduce_sum(kernel)
    kernel = tf.tile(kernel[:, :, tf.newaxis, tf.newaxis], [1, 1, 3, 1])

    blurred = tf.nn.depthwise_conv2d(img[tf.newaxis, ...], kernel,
                                     [1, 1, 1, 1], 'SAME')
    return blurred[0]


def downscale(img):
    """Throw away resolution and put it back — a distant subject on a phone."""
    factor = tf.random.uniform([], 0.25, 0.6)
    small = tf.cast(tf.round(IMAGE_SIZE * factor), tf.int32)
    low = tf.image.resize(img, (small, small))
    return tf.image.resize(low, (IMAGE_SIZE, IMAGE_SIZE))


def jpeg_artifacts(img):
    """Re-encode hard. Phone photographs arrive compressed, not pristine."""
    as_bytes = tf.cast(tf.clip_by_value(img, 0.0, 255.0), tf.uint8)
    as_bytes = tf.image.random_jpeg_quality(as_bytes, 30, 75)
    return tf.cast(as_bytes, tf.float32)


def degrade(img):
    """Apply at most one camera degradation, so they do not compound."""
    def apply_one():
        index = tf.random.uniform([], 0, 4, dtype=tf.int32)
        return tf.switch_case(index, [
            lambda: gaussian_blur(img), lambda: motion_blur(img),
            lambda: downscale(img), lambda: jpeg_artifacts(img),
        ])

    return tf.cond(tf.random.uniform([]) < DEGRADE_PROBABILITY,
                   apply_one, lambda: img)


def random_erasing(img):
    """Blank a rectangle — a hand, a phone, or a passer-by crossing the frame.

    Occlusion robustness matters here specifically because the cigarette is a
    small object: if the model has learned to depend on one region of the
    frame, erasing part of it during training forces it to look elsewhere.
    """
    def erase():
        area = IMAGE_SIZE * IMAGE_SIZE * tf.random.uniform([], 0.02, 0.15)
        aspect = tf.exp(tf.random.uniform([], tf.math.log(0.3), tf.math.log(3.3)))

        height = tf.clip_by_value(tf.cast(tf.sqrt(area / aspect), tf.int32),
                                  1, IMAGE_SIZE)
        width = tf.clip_by_value(tf.cast(tf.sqrt(area * aspect), tf.int32),
                                 1, IMAGE_SIZE)
        top = tf.random.uniform([], 0, IMAGE_SIZE - height + 1, tf.int32)
        left = tf.random.uniform([], 0, IMAGE_SIZE - width + 1, tf.int32)

        rows = tf.range(IMAGE_SIZE)[:, tf.newaxis]
        cols = tf.range(IMAGE_SIZE)[tf.newaxis, :]
        inside = ((rows >= top) & (rows < top + height)
                  & (cols >= left) & (cols < left + width))
        mask = tf.cast(inside, tf.float32)[..., tf.newaxis]

        noise = tf.random.uniform(tf.shape(img), 0.0, 255.0)
        return img * (1.0 - mask) + noise * mask

    return tf.cond(tf.random.uniform([]) < ERASE_PROBABILITY, erase, lambda: img)


def augment(img, label):
    img = random_resized_crop(img)          # geometry / subject distance
    img = tf.image.random_flip_left_right(img)
    img = randaugment(img)                  # colour and tone
    img = degrade(img)                      # phone-camera conditions
    img = random_erasing(img)               # occlusion
    return tf.clip_by_value(img, 0.0, 255.0), label


SHUFFLE_SEED = 20260813


def build(split, training, seed=None):
    """`seed` overrides the shuffle seed — used by the seed study in section 11.

    Left as None everywhere else so the headline run stays reproducible.
    """
    rows = manifest[manifest['split'] == split]
    paths = [str(ML_DIR / p) for p in rows['path']]
    labels = [CLASS_NAMES.index(l) for l in rows['label']]

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SHUFFLE_SEED if seed is None else seed,
                        reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE)
    # Evaluation splits are NEVER shuffled and NEVER augmented: score_split in
    # cell 18 relies on their order matching the manifest.
    ds = ds.map(augment if training else resize_only, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = build('train', training=True)
val_ds   = build('val',   training=False)
test_ds  = build('test',  training=False)

images, labels = next(iter(train_ds))
print('batch', images.shape, 'range', float(tf.reduce_min(images)),
      '-', float(tf.reduce_max(images)))
print('labels', labels.shape, '(one-hot)  first row', labels[0].numpy())

## 4. Model

A ViT pre-trained on ImageNet-21k, with a fresh 2-class head. The dataset here
is far too small to train a transformer from scratch — transfer learning is what
makes this feasible at all, and is the approach Mazurets et al. (2022) used for
the same task.

This cell also defines **what training measures**. `metrics=['accuracy']` and
early stopping on `val_loss` kept whichever checkpoint had the lowest loss,
which on a precision-critical task is not the checkpoint worth keeping.

In [ ]:
PRESET = 'vit_base_patch16_224_imagenet21k'

# activation='softmax' is REQUIRED and is not the default.
#
# KerasHub ships activation=None, which makes the model output raw LOGITS —
# unbounded numbers that can be negative. Pairing that with
# SparseCategoricalCrossentropy(from_logits=False) silently trains against a
# broken objective: the loss treats negative logits as probabilities, learns
# almost nothing, and predict() then returns negative 'confidence' scores that
# no threshold can work with.
#
# Two consistent options exist. This notebook takes the first:
#   activation='softmax'  +  from_logits=False   <- outputs are probabilities
#   activation=None       +  from_logits=True    <- outputs are logits
classifier = keras_hub.models.ViTImageClassifier.from_preset(
    PRESET,
    num_classes=2,
    activation='softmax',
)

# Stage 1: freeze the backbone. The new head starts random, and letting large
# random gradients flow into pre-trained weights destroys the features that
# make transfer learning work.
classifier.backbone.trainable = False

# ---------------------------------------------------------------------------
# What training measures
#
# The previous compile used metrics=['accuracy'] and early-stopped on
# val_loss. Neither is precision, so the checkpoint that survived was the one
# with the best loss — not the one that serves S10 best. On a task where a
# false positive reaches enforcement uncorrected and a false negative only
# asks for a retake, that is measuring the wrong thing.
#
# The metrics below read the POSITIVE-class column, so they answer the
# question the report asks: of the photographs we accept, how many are real.
#
# Early stopping uses PR-AUC rather than precision-at-0.5. PR-AUC is
# threshold-independent, so it does not quietly re-decide the operating point
# that ml/select_threshold.py selects on validation.
# ---------------------------------------------------------------------------


class _PositiveColumn:
    """Restrict a binary metric to P(smoking).

    Keras's Precision and Recall compare the whole output against the whole
    target. With a 2-column softmax that silently averages the positive class
    with its own complement, which is not the quantity Chapter 5 reports.
    """

    def update_state(self, y_true, y_pred, sample_weight=None):
        return super().update_state(
            y_true[:, POSITIVE_INDEX],
            y_pred[:, POSITIVE_INDEX],
            sample_weight=sample_weight,
        )


class PositivePrecision(_PositiveColumn, keras.metrics.Precision):
    pass


class PositiveRecall(_PositiveColumn, keras.metrics.Recall):
    pass


class PositivePRAUC(_PositiveColumn, keras.metrics.AUC):
    pass


def training_metrics():
    """Fresh metric objects per compile — they carry state across epochs."""
    return [
        'accuracy',
        PositivePrecision(name='precision'),
        PositiveRecall(name='recall'),
        PositivePRAUC(curve='PR', name='pr_auc'),
    ]


# Label smoothing turns the hard targets 0/1 into 0.05/0.95, which stops the
# model driving its logits arbitrarily far apart to reduce a loss that is
# already near zero. The current score distribution — mean 0.939 for smoking
# against 0.055 for not — is exactly the over-confidence it tempers, and a
# less extreme distribution is easier to calibrate afterwards.
#
# It is applied INSIDE the loss, so the metrics above still see clean 0/1
# targets and remain honest.
#
# Keras implements label smoothing only on CategoricalCrossentropy, which is
# why cell 8 emits one-hot labels rather than integer indices.
LABEL_SMOOTHING = 0.1

# Epoch counts live here, with the rest of the training configuration, rather
# than inside the fit cells. The seed study in section 11 trains from scratch
# and needs them; defining them there too would let the two drift apart, and
# defining them only in the fit cells would force a full training run before
# the seed study could start.
EPOCHS_HEAD = 5
EPOCHS_FINE = 10


def training_loss():
    return keras.losses.CategoricalCrossentropy(
        from_logits=False, label_smoothing=LABEL_SMOOTHING)


classifier.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=training_loss(),
    metrics=training_metrics(),
)

# Fail loudly here rather than discovering it after a full training run.
_probe = classifier.predict(val_ds.take(1), verbose=0)
assert _probe.min() >= 0.0 and _probe.max() <= 1.0, (
    f'Model is emitting logits, not probabilities (range {_probe.min():.3f} to '
    f'{_probe.max():.3f}). Check the activation argument above.')
import numpy as _np
assert _np.allclose(_probe.sum(axis=1), 1.0, atol=1e-3), 'Rows should sum to 1.'
print(f'Output check passed: probabilities in [{_probe.min():.3f}, {_probe.max():.3f}]')
classifier.summary()


## 5. Stage 1 — train the head


In [ ]:
history_head = classifier.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=[
        # mode='max' is required and is not inferred reliably for a custom
        # metric name: with the default 'auto' on an unrecognised name, Keras
        # can minimise PR-AUC and restore the WORST weights it saw.
        keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max',
                                      patience=3, restore_best_weights=True),
    ],
)

## 6. Stage 2 — fine-tune

Unfreeze the backbone at a much lower learning rate. 1e-3 would wreck the
pre-trained weights; 1e-5 adapts them.


In [ ]:
classifier.backbone.trainable = True

# Fresh metric objects: the ones from cell 10 carry state from stage 1.
classifier.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=training_loss(),
    metrics=training_metrics(),
)

history_fine = classifier.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max',
                                      patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc', mode='max',
                                          factor=0.3, patience=2),
    ],
)

## 7. Training curves

Chapter 5 needs these. A validation loss that rises while training loss keeps
falling is overfitting — collect more data or stop earlier.

PR-AUC is the panel to read first: it is what early stopping acts on, and
unlike accuracy it does not flatter a model on a near-balanced split. Training
metrics are measured on augmented images and validation metrics on clean ones,
so the training curves sitting below the validation curves is expected here,
not a sign of a bug.

In [ ]:
import matplotlib.pyplot as plt

def series(key):
    return list(history_head.history[key]) + list(history_fine.history[key])

panels = [('pr_auc', 'val_pr_auc', 'PR-AUC  (early stopping acts on this)'),
          ('precision', 'val_precision', 'Precision @ 0.5'),
          ('recall', 'val_recall', 'Recall @ 0.5'),
          ('loss', 'val_loss', 'Loss  (label-smoothed)')]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (train_key, val_key, title) in zip(axes.flat, panels):
    ax.plot(series(train_key), label='train (augmented)')
    ax.plot(series(val_key), label='validation (clean)')
    ax.axvline(len(history_head.history[train_key]) - 0.5, ls='--', c='grey', lw=1)
    ax.set_title(title + '\n(dashed = backbone unfrozen)', fontsize=10)
    ax.set_xlabel('epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

best = max(series('val_pr_auc'))
print(f'Best validation PR-AUC {best:.4f} at epoch '
      f'{series("val_pr_auc").index(best) + 1}')

## 8. Score the validation and test splits

Two splits, two different jobs, and the order matters.

**Validation** is where the confidence threshold gets chosen. It has already
influenced training through early stopping, so it cannot carry the reported
figure — but that is exactly what makes it the right place to fit one more
parameter.

**Test** has not influenced training, early stopping, or the threshold. It is
touched once, to report metrics at the threshold already frozen on validation.

Selecting the threshold on test and then reporting test metrics at that point
fits the operating point to the same images the result is claimed on, which
biases the reported precision upward. See `ml/NEXT-STEPS.md` section 1.

The CSVs written here are a sanity check. **The authoritative scores come from
`ml/score_split.py`**, which runs the images through
`backend/app/services/inference.py` — the code that actually serves S10.
Preprocessing differences between this notebook and the backend have shifted
individual scores by up to 0.46 before.

In [ ]:
scored = classifier.evaluate(test_ds, return_dict=True, verbose=0)
for name in ('loss', 'accuracy', 'precision', 'recall', 'pr_auc'):
    print(f'test {name:>10s}  {scored[name]:.4f}')

# These are measured at Keras's fixed 0.5 cutoff, which is NOT the operating
# point. PR-AUC is the threshold-independent figure and the one to compare
# between runs; the precision and recall that go in Chapter 5 come from
# select_threshold.py, at the threshold chosen on validation.
print('\nPrecision and recall above are at 0.5, not at the operating point.')


def score_split(dataset, split):
    """Write label,score for one split, in manifest order.

    The dataset is built with training=False, so it is not shuffled and rows
    line up with the manifest. Any change to build() that shuffles an
    evaluation split silently misaligns labels from scores.
    """
    probs = classifier.predict(dataset, verbose=0)
    positive = probs[:, POSITIVE_INDEX]

    # select_threshold.py sweeps thresholds between 0 and 1, so anything
    # outside that range makes every threshold reject everything.
    assert positive.min() >= 0.0 and positive.max() <= 1.0, (
        f'Scores outside [0,1]: {positive.min():.3f} to {positive.max():.3f}. '
        'The model is emitting logits — re-run the model cell with softmax.')

    rows = manifest[manifest['split'] == split].reset_index(drop=True)
    assert len(rows) == len(positive), (
        f'{len(rows)} manifest rows but {len(positive)} scores for {split}.')

    frame = pd.DataFrame({'label': rows['label'], 'score': positive})
    path = f'/content/{split}_scores.csv'
    frame.to_csv(path, index=False)

    gap = (frame[frame.label == 'smoking']['score'].mean()
           - frame[frame.label == 'notsmoking']['score'].mean())
    print(f'{split}: {len(frame)} images, mean score gap {gap:.3f} '
          f'({"healthy" if gap > 0.3 else "WEAK — see notes below"})')
    return path


val_path = score_split(val_ds, 'val')
test_path = score_split(test_ds, 'test')

files.download(val_path)
files.download(test_path)
files.download('/content/training_curves.png')

print('\nThreshold: select on val_scores.csv, then report on test_scores.csv')
print('at that frozen value. Re-score through the backend first — see section 10.')

## 9. Export the model

Saved in Keras 3 format and loaded by `backend/app/services/inference.py`.
Download it, unzip into `ml/artifacts/`, and set `AI_MODEL_PATH` in
`backend/.env`.

Record `MODEL_VERSION` in `backend/.env` too — `AI_CLASSIFICATION.model_version`
stores it per report, so a later retrain never silently changes the meaning of
a historical confidence score.


In [ ]:
# Bump this on EVERY retrain. AI_CLASSIFICATION.model_version stores it per
# report, so a stale version silently changes what a historical confidence
# score meant. This run uses the new augmentation, metrics and loss, so it is
# not comparable to vit-scope-v1 and must not reuse its name.
MODEL_VERSION = 'vit-scope-v2'

classifier.save(f'/content/{MODEL_VERSION}.keras')

metadata = {
    'model_version': MODEL_VERSION,
    'preset': PRESET,
    'image_size': IMAGE_SIZE,
    'class_names': CLASS_NAMES,
    'positive_index': CLASS_NAMES.index('smoking'),
    'test_metrics_at_0.5': {k: float(v) for k, v in scored.items()},
    'label_smoothing': LABEL_SMOOTHING,
    'augmentation': 'random_resized_crop+flip+randaugment+degrade+erasing',
    'train_images': int((manifest['split'] == 'train').sum()),
    'val_images': int((manifest['split'] == 'val').sum()),
    'test_images': int((manifest['split'] == 'test').sum()),
}
pathlib.Path('/content/model_metadata.json').write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

files.download(f'/content/{MODEL_VERSION}.keras')
files.download('/content/model_metadata.json')


## 10. Next

1. Put `vit-scope-v1.keras` in `ml/artifacts/` and point `AI_MODEL_PATH` at it
2. Re-score both splits through the backend's own inference code — not the
   CSVs downloaded above, which came from this notebook's pipeline:

   ```bash
   ./backend/.venv/bin/python ml/score_split.py val
   ./backend/.venv/bin/python ml/score_split.py test
   ```

3. **Select** the threshold on validation, and freeze what it returns:

   ```bash
   ./backend/.venv/bin/python ml/select_threshold.py \
       ml/artifacts/val_scores_backend.csv
   ```

4. **Report** on test at that frozen threshold, changing nothing afterwards:

   ```bash
   ./backend/.venv/bin/python ml/select_threshold.py \
       ml/artifacts/test_scores_backend.csv --at-threshold <frozen>
   ```

5. Set `AI_CONFIDENCE_THRESHOLD` and `AI_MODEL_VERSION` in `backend/.env`

Steps 3 and 4 are not interchangeable. If the threshold is adjusted after
seeing step 4's numbers, the test split has been spent and the figure is no
longer an independent estimate.

Keep `training_curves.png`, the composition table from `ml/data/report.md`, and
both threshold outputs — together they are the evaluation evidence for
Chapter 5.

## 11. Seed study — "one run is not a result"

**Optional, and roughly three times the runtime of everything above.** Run it
only once the sections above have produced a model you are happy with.

Every figure so far comes from one training run. But the new head starts from
random weights, the images are shuffled in a random order, and the augmentation
crops and blurs at random — so a second run with a different seed produces a
slightly different model. On 293 test images the difference between 3 false
positives and 6 is one unlucky initialisation.

This section trains the **same recipe on the same split** three times, changing
only the seed, and writes per-seed scores. `ml/aggregate_seeds.py` then turns
them into `mean ± standard deviation`, which is what the report should quote.

Two things it deliberately does NOT do:

- **It does not replace `vit-scope-v2`.** That model, scored through the
  backend, stays the headline. This measures spread, not the headline.
- **It does not export three models.** Three ViT checkpoints is ~3 GB of
  downloads. Only the scores come home.

Because these scores come from this notebook's pipeline rather than the
backend's, the absolute values differ very slightly from the backend-scored
figures. That is fine for measuring variance — every seed is scored the same
way — but say so in the report rather than mixing the two sets of numbers.

In [ ]:
import gc

# Section 4 leaves a built ViT in memory. A ViT-Base is about a gigabyte, and
# two of them do not fit alongside a training run, so release it before the
# first seed if this notebook has run that far.
if 'classifier' in dir():
    del classifier
    keras.backend.clear_session()
    gc.collect()
    print('released the section 4 model')

SEEDS = [1, 2, 3]
SEED_DIR = pathlib.Path('/content/seeds')
SEED_DIR.mkdir(exist_ok=True)


def train_one_seed(seed):
    """Train the recipe from scratch at `seed` and write its val/test scores.

    keras.utils.set_random_seed covers Python, NumPy and TensorFlow in one
    call. Everything else — the split, the architecture, the epochs, the
    augmentation settings — is held fixed, so the spread across seeds is
    attributable to training randomness and nothing else.
    """
    keras.utils.set_random_seed(seed)

    train = build('train', training=True, seed=seed)
    val = build('val', training=False)
    test = build('test', training=False)

    model = keras_hub.models.ViTImageClassifier.from_preset(
        PRESET, num_classes=NUM_CLASSES, activation='softmax')

    # Stage 1 — head only.
    model.backbone.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss=training_loss(), metrics=training_metrics())
    model.fit(train, validation_data=val, epochs=EPOCHS_HEAD, verbose=2,
              callbacks=[keras.callbacks.EarlyStopping(
                  monitor='val_pr_auc', mode='max', patience=3,
                  restore_best_weights=True)])

    # Stage 2 — unfreeze.
    model.backbone.trainable = True
    model.compile(optimizer=keras.optimizers.Adam(1e-5),
                  loss=training_loss(), metrics=training_metrics())
    model.fit(train, validation_data=val, epochs=EPOCHS_FINE, verbose=2,
              callbacks=[
                  keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max',
                                                patience=3,
                                                restore_best_weights=True),
                  keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc',
                                                    mode='max', factor=0.3,
                                                    patience=2),
              ])

    for split, dataset in (('val', val), ('test', test)):
        probs = model.predict(dataset, verbose=0)[:, POSITIVE_INDEX]
        rows = manifest[manifest['split'] == split].reset_index(drop=True)
        assert len(rows) == len(probs), f'{split}: {len(rows)} rows, {len(probs)} scores'
        pd.DataFrame({'label': rows['label'], 'score': probs}).to_csv(
            SEED_DIR / f'{split}_scores_seed{seed}.csv', index=False)

    scored = model.evaluate(test, return_dict=True, verbose=0)
    print(f"seed {seed}: test pr_auc {scored['pr_auc']:.4f}  "
          f"precision@0.5 {scored['precision']:.4f}  recall@0.5 {scored['recall']:.4f}")

    # A ViT-Base is ~1 GB in memory. Without this the second seed runs out.
    del model
    keras.backend.clear_session()
    gc.collect()


for seed in SEEDS:
    print(f'\n{"=" * 60}\nSEED {seed}\n{"=" * 60}')
    train_one_seed(seed)

print('\nDone. Unzip into ml/artifacts/seeds/ and run:')
print('  ./backend/.venv/bin/python ml/aggregate_seeds.py')

!cd /content && zip -qr seed_scores.zip seeds
files.download('/content/seed_scores.zip')